# Ingested CSV information to bronze layer.


## 1. Import csv config data

In [0]:

import bronze_config as bc  # import bronze config
import importlib            # import reload bronze config

importlib.reload(bc)        # reload bronze config

## 2. Created ingestion scripts

In [0]:
def load_bronze(config):

    load_bronze_res = "OK"

    bronze_df = (
                    spark.read
                        .format(config["format"])
                        .option("header", True)
                        .option("inferSchema", True)
                        .load(f"{config["path"]}/{config["file"]}.{config["format"]}")
    )

    (
        bronze_df.write
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{config["table_path"]}.{config["table"]}")  
    )
    
    # check existing table after completed ingestion
    if spark.table(f"{config["table_path"]}.{config["table"]}").count() != 0:
        load_bronze_res = "OK"
    else:
        load_bronze_res = "NG"
                        
    return load_bronze_res

In [0]:
# bronze layer data ingestion process
try:
    for config in bc.BRONZE_CONFIG:
        
        load_bronze_res = load_bronze(config)

        if load_bronze_res == "OK":
            print("Load data successfully")
            # move completed write file from landing to processed folder
        else:
            print("Error : ")
except Exception as e:
    print("Error : ", e)